<a href="https://colab.research.google.com/github/YukthaGowda/AUTOIMMUNE-DISEASE/blob/main/BIM_Decision_System_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ BIM-AI Decision Making System
### Step 1: 5 Charts + Feature Importance (inline Colab) → Step 2: Launch Interactive Streamlit Dashboard

**Dashboard tabs:**
- 📊 Portfolio Overview — charts, feature importance, top-risk table
- 🗺️ Geospatial Map — project bubbles on USA map, city-level summaries  
- 🔬 Project Simulator — sliders + risk prediction + Ollama decision plan
- 🤖 AI Advisor — portfolio-level questions to Ollama


## Cell 1 — Install

In [ ]:
!pip install -q plotly scikit-learn streamlit pyngrok requests
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 72.8 MB/s eta 0:00:00
Done


## Cell 2 — Install & Start Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama','serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull llama3.2
print('Ollama ready')

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

## Cell 3 — Upload Dataset

## Cell 4 — Load Data

In [ ]:
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('bim_ai_civil_engineering_dataset.csv')
df['Overrun_Pct']    = df['Cost_Overrun'] / df['Planned_Cost'] * 100
df['Schedule_Ratio'] = df['Actual_Duration'] / df['Planned_Duration']
df['Overrun_Abs']    = df['Cost_Overrun'].abs()
df['Risk_Num']       = df['Risk_Level'].map({'Low':0,'Medium':1,'High':2})
RISK_COLORS = {'High':'#e74c3c','Medium':'#f39c12','Low':'#27ae60'}
print('Loaded', len(df), 'projects |', df['Risk_Level'].value_counts().to_dict())
df.head(3)

---
## Chart 1 — Risk Level by Project Type

In [ ]:
rt = df.groupby(['Project_Type','Risk_Level']).size().reset_index(name='Count')
rt['Pct'] = rt['Count'] / rt.groupby('Project_Type')['Count'].transform('sum') * 100
fig = px.bar(rt, x='Project_Type', y='Pct', color='Risk_Level',
             barmode='stack', color_discrete_map=RISK_COLORS,
             title='Which project types carry the most risk?',
             labels={'Pct':'Share of Projects (%)','Project_Type':''},
             text=rt['Pct'].round(1).astype(str)+'%')
fig.update_traces(textposition='inside', textfont_size=11)
fig.update_layout(height=420, title_font_size=17)
fig.show()

---
## Chart 2 — Cost Overrun by City

In [ ]:
city = df.groupby('Location')['Overrun_Pct'].mean().reset_index().sort_values('Overrun_Pct', ascending=False)
fig = px.bar(city, x='Location', y='Overrun_Pct',
             color='Overrun_Pct', color_continuous_scale=['#27ae60','#f39c12','#e74c3c'],
             title='Average Cost Overrun % by City',
             labels={'Overrun_Pct':'Avg Cost Overrun (%)','Location':''},
             text_auto='.1f')
fig.update_layout(height=420, title_font_size=17, coloraxis_showscale=False)
fig.show()

---
## Chart 3 — Schedule Delay Distribution

In [ ]:
fig = px.histogram(df, x='Schedule_Deviation', color='Risk_Level',
                   nbins=30, barmode='overlay', opacity=0.78,
                   color_discrete_map=RISK_COLORS,
                   title='How many days are projects delayed?',
                   labels={'Schedule_Deviation':'Days Behind Schedule'})
fig.add_vline(x=0, line_dash='dash', line_color='black', line_width=2,
              annotation_text=' On time', annotation_position='top right')
fig.update_layout(height=420, title_font_size=17)
fig.show()

---
## Chart 4 — Correlation Heatmap

In [ ]:
key_cols = ['Cost_Overrun','Schedule_Deviation','Safety_Risk_Score',
            'Accident_Count','Vibration_Level','Equipment_Utilization',
            'Labor_Hours','Completion_Percentage','Risk_Num','Overrun_Pct']
corr = df[key_cols].corr().round(2)
fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=corr.columns.tolist(), y=corr.index.tolist(),
    colorscale='RdBu_r', zmid=0,
    text=corr.values, texttemplate='%{text:.2f}', textfont=dict(size=11),
    colorbar=dict(title='r')
))
fig.update_layout(title='What variables are related?',
                  title_font_size=17, height=520, xaxis=dict(tickangle=-40))
fig.show()

---
## Chart 5 — Completion % vs Safety Risk

In [ ]:
fig = px.scatter(df, x='Safety_Risk_Score', y='Completion_Percentage',
                 color='Risk_Level', color_discrete_map=RISK_COLORS,
                 size='Overrun_Abs', size_max=16, facet_col='Project_Type',
                 hover_data=['Project_ID','Location','Schedule_Deviation'],
                 title='Completion % vs Safety Risk  (bubble = cost overrun size)',
                 labels={'Safety_Risk_Score':'Safety Risk Score',
                         'Completion_Percentage':'Completion (%)'})
fig.update_layout(height=400, title_font_size=17)
fig.show()

---
## Feature Importance — What predicts Risk?
*Red bars = variables to focus on and control*

In [ ]:
features = ['Cost_Overrun','Schedule_Deviation','Vibration_Level','Crack_Width',
            'Accident_Count','Safety_Risk_Score','Completion_Percentage',
            'Equipment_Utilization','Labor_Hours','Overrun_Pct','Schedule_Ratio',
            'Humidity','Temperature','Air_Quality_Index','Energy_Consumption']
X = df[features].fillna(0)
y = LabelEncoder().fit_transform(df['Risk_Level'])
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X, y)
imp = pd.Series(rf.feature_importances_, index=features).sort_values()
bar_colors = ['#e74c3c' if v > float(imp.median()) else '#95a5a6' for v in imp.values]
fig = go.Figure(go.Bar(
    x=imp.values, y=imp.index, orientation='h',
    marker_color=bar_colors,
    text=[str(round(v,3)) for v in imp.values], textposition='outside'
))
fig.update_layout(title='Risk Drivers — Red = most important to monitor',
                  title_font_size=16, height=520,
                  xaxis_title='Importance Score', margin=dict(l=180,r=100))
fig.show()
print('Top 5:')
for i,(k,v) in enumerate(imp.sort_values(ascending=False).head(5).items(),1):
    print(' ', i, k, round(v,4))

---
## Write Dashboard File
*Writes `dashboard.py` to Colab disk — includes the Geospatial Map tab*

In [ ]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import requests

st.set_page_config(page_title="BIM-AI Decision System", page_icon="🏗️", layout="wide")
RISK_COLORS = {'High':'#e74c3c','Medium':'#f39c12','Low':'#27ae60'}

# City coordinates
CITY_COORDS = {
    'New York':    {'lat': 40.7128, 'lon': -74.0060},
    'Los Angeles': {'lat': 34.0522, 'lon': -118.2437},
    'Chicago':     {'lat': 41.8781, 'lon': -87.6298},
    'Houston':     {'lat': 29.7604, 'lon': -95.3698},
    'Seattle':     {'lat': 47.6062, 'lon': -122.3321},
}

@st.cache_data
def load(f=None):
    df = pd.read_csv(f if f else 'bim_ai_civil_engineering_dataset.csv')
    df['Overrun_Pct']    = df['Cost_Overrun'] / df['Planned_Cost'] * 100
    df['Schedule_Ratio'] = df['Actual_Duration'] / df['Planned_Duration']
    df['Overrun_Abs']    = df['Cost_Overrun'].abs()
    df['Risk_Num']       = df['Risk_Level'].map({'Low':0,'Medium':1,'High':2})
    df['lat']            = df['Location'].map(lambda c: CITY_COORDS[c]['lat'])
    df['lon']            = df['Location'].map(lambda c: CITY_COORDS[c]['lon'])
    # small jitter so bubbles don't perfectly overlap
    np.random.seed(42)
    df['lat'] = df['lat'] + np.random.uniform(-0.18, 0.18, len(df))
    df['lon'] = df['lon'] + np.random.uniform(-0.18, 0.18, len(df))
    return df

@st.cache_resource
def train_model(df):
    feats = ['Cost_Overrun','Schedule_Deviation','Vibration_Level','Crack_Width',
             'Accident_Count','Safety_Risk_Score','Completion_Percentage',
             'Equipment_Utilization','Labor_Hours','Overrun_Pct','Schedule_Ratio']
    X = df[feats].fillna(0)
    y = LabelEncoder().fit_transform(df['Risk_Level'])
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X, y)
    return rf, feats

def ask_ollama(prompt, model='llama3.2'):
    try:
        r = requests.post('http://localhost:11434/api/generate',
                          json={'model':model,'prompt':prompt,'stream':False}, timeout=90)
        if r.status_code == 200:
            return r.json().get('response','No response.')
        return "Ollama error " + str(r.status_code)
    except Exception as e:
        return "Ollama not running. In Colab run: !ollama serve &"

# ── Sidebar ────────────────────────────────────────────────────
with st.sidebar:
    st.title("BIM-AI")
    st.caption("Decision Making System")
    st.markdown("---")
    uploaded = st.file_uploader("Upload CSV", type=['csv'])
    df = load(uploaded)
    model, FEATS = train_model(df)
    st.markdown("### Filters")
    sel_type = st.multiselect("Project Type", sorted(df['Project_Type'].unique()), default=sorted(df['Project_Type'].unique()))
    sel_city = st.multiselect("City", sorted(df['Location'].unique()), default=sorted(df['Location'].unique()))
    sel_risk = st.multiselect("Risk Level", ['High','Medium','Low'], default=['High','Medium','Low'])
    st.markdown("---")
    ollama_model = st.text_input("Ollama model", value="llama3.2")

fdf = df[
    df['Project_Type'].isin(sel_type) &
    df['Location'].isin(sel_city) &
    df['Risk_Level'].isin(sel_risk)
].copy()

st.title("BIM-AI Decision Making System")
st.caption("Showing " + str(len(fdf)) + " of " + str(len(df)) + " projects")

tab1, tab2, tab3, tab4 = st.tabs([
    "📊 Portfolio Overview",
    "🗺️ Geospatial Map",
    "🔬 Project Simulator",
    "🤖 AI Advisor"
])

# ══════════════════════════════════════════════════════════════
# TAB 1 — PORTFOLIO OVERVIEW
# ══════════════════════════════════════════════════════════════
with tab1:
    high = fdf[fdf['Risk_Level']=='High']
    c1,c2,c3,c4,c5 = st.columns(5)
    c1.metric("Total Projects", str(len(fdf)))
    c2.metric("High Risk", str(len(high)), str(round(len(high)/max(len(fdf),1)*100,1))+"%", delta_color='inverse')
    c3.metric("Avg Overrun", "$" + str(round(fdf['Cost_Overrun'].mean()/1e6,2)) + "M", delta_color='inverse')
    c4.metric("Avg Delay", str(round(fdf['Schedule_Deviation'].mean(),0))+" days", delta_color='inverse')
    c5.metric("Avg Completion", str(round(fdf['Completion_Percentage'].mean(),1))+"%")

    st.markdown("---")
    col1, col2 = st.columns(2)
    with col1:
        st.markdown("##### Risk by project type")
        rt = fdf.groupby(['Project_Type','Risk_Level']).size().reset_index(name='Count')
        rt['Pct'] = rt['Count'] / rt.groupby('Project_Type')['Count'].transform('sum') * 100
        fig = px.bar(rt, x='Project_Type', y='Pct', color='Risk_Level', barmode='stack',
                     color_discrete_map=RISK_COLORS, labels={'Pct':'%','Project_Type':''})
        fig.update_layout(height=300, margin=dict(t=5,b=5), legend=dict(orientation='h',y=-0.3))
        st.plotly_chart(fig, use_container_width=True)
    with col2:
        st.markdown("##### Cost overrun % by city")
        city = fdf.groupby('Location')['Overrun_Pct'].mean().reset_index().sort_values('Overrun_Pct',ascending=False)
        fig2 = px.bar(city, x='Location', y='Overrun_Pct',
                      color='Overrun_Pct', color_continuous_scale=['#27ae60','#f39c12','#e74c3c'],
                      labels={'Overrun_Pct':'Avg Overrun (%)','Location':''}, text_auto='.1f')
        fig2.update_layout(height=300, margin=dict(t=5,b=5), coloraxis_showscale=False)
        st.plotly_chart(fig2, use_container_width=True)

    col3, col4 = st.columns(2)
    with col3:
        st.markdown("##### Schedule delay distribution")
        fig3 = px.histogram(fdf, x='Schedule_Deviation', color='Risk_Level', nbins=28,
                            barmode='overlay', opacity=0.78, color_discrete_map=RISK_COLORS,
                            labels={'Schedule_Deviation':'Days Behind'})
        fig3.add_vline(x=0, line_dash='dash', line_color='black')
        fig3.update_layout(height=300, margin=dict(t=5,b=5), legend=dict(orientation='h',y=-0.3))
        st.plotly_chart(fig3, use_container_width=True)
    with col4:
        st.markdown("##### Completion % vs Safety Risk")
        fig4 = px.scatter(fdf, x='Safety_Risk_Score', y='Completion_Percentage',
                          color='Risk_Level', size='Overrun_Abs', size_max=14,
                          color_discrete_map=RISK_COLORS,
                          hover_data=['Project_ID','Project_Type','Location'],
                          labels={'Safety_Risk_Score':'Safety Risk','Completion_Percentage':'Completion (%)'})
        fig4.update_layout(height=300, margin=dict(t=5,b=5), legend=dict(orientation='h',y=-0.3))
        st.plotly_chart(fig4, use_container_width=True)

    st.markdown("---")
    st.markdown("##### Feature Importance — Top risk drivers (red = focus here)")
    imp = pd.Series(model.feature_importances_, index=FEATS).sort_values()
    bar_colors = ['#e74c3c' if v > float(imp.median()) else '#95a5a6' for v in imp.values]
    fig5 = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation='h',
                            marker_color=bar_colors,
                            text=[str(round(v,3)) for v in imp.values], textposition='outside'))
    fig5.update_layout(height=360, margin=dict(t=5,b=5,l=160,r=80), xaxis_title='Importance')
    st.plotly_chart(fig5, use_container_width=True)

    st.markdown("---")
    st.markdown("##### Top 10 High-Risk Projects")
    top10 = (fdf[fdf['Risk_Level']=='High'].sort_values('Cost_Overrun',ascending=False)
               .head(10)[['Project_ID','Project_Type','Location','Cost_Overrun',
                           'Schedule_Deviation','Safety_Risk_Score','Completion_Percentage']]
               .reset_index(drop=True))
    top10['Cost_Overrun'] = top10['Cost_Overrun'].apply(lambda x: '$'+str(round(x/1e6,2))+'M')
    top10['Schedule_Deviation'] = top10['Schedule_Deviation'].apply(lambda x: str(round(x,0))+' days')
    st.dataframe(top10, use_container_width=True, hide_index=True,
                 column_config={'Safety_Risk_Score': st.column_config.ProgressColumn(
                     'Safety Risk', min_value=0, max_value=10, format='%.1f')})

# ══════════════════════════════════════════════════════════════
# TAB 2 — GEOSPATIAL MAP
# ══════════════════════════════════════════════════════════════
with tab2:
    st.markdown("### Geospatial Project Risk Map")
    st.caption("Each bubble = one project. Size = cost overrun. Colour = risk level. Hover for details.")

    map_col1, map_col2 = st.columns([3, 1])

    with map_col2:
        st.markdown("#### Map Options")
        map_metric = st.selectbox("Bubble size = ", [
            "Cost Overrun ($)",
            "Schedule Deviation (days)",
            "Safety Risk Score",
            "Accident Count",
            "Completion %",
        ])
        map_color = st.selectbox("Colour by", ["Risk Level", "Project Type"])
        map_style = st.selectbox("Map style", [
            "open-street-map",
            "carto-positron",
            "carto-darkmatter",
        ])
        show_city_summary = st.checkbox("Show city summary layer", value=True)

        metric_col = {
            "Cost Overrun ($)":           "Overrun_Abs",
            "Schedule Deviation (days)":  "Schedule_Deviation",
            "Safety Risk Score":          "Safety_Risk_Score",
            "Accident Count":             "Accident_Count",
            "Completion %":               "Completion_Percentage",
        }[map_metric]

        # City-level summary for stats panel
        city_summary = fdf.groupby('Location').agg(
            Projects=('Project_ID','count'),
            High_Risk=('Risk_Num', lambda x: (fdf.loc[x.index,'Risk_Level']=='High').sum()),
            Avg_Overrun=('Overrun_Pct','mean'),
            Avg_Delay=('Schedule_Deviation','mean'),
            Avg_Completion=('Completion_Percentage','mean'),
            Total_Accidents=('Accident_Count','sum'),
        ).reset_index().round(1)

        st.markdown("#### City Summary")
        for _, row in city_summary.sort_values('Avg_Overrun', ascending=False).iterrows():
            risk_pct = round(row['High_Risk']/row['Projects']*100,0)
            color = '#e74c3c' if risk_pct > 40 else ('#f39c12' if risk_pct > 25 else '#27ae60')
            st.markdown(
                f"**{row['Location']}**  "
                f"<span style='color:{color}'>■</span> "
                f"{int(row['High_Risk'])}/{int(row['Projects'])} high-risk<br>"
                f"Overrun: {row['Avg_Overrun']}% | Delay: {row['Avg_Delay']}d",
                unsafe_allow_html=True
            )
            st.markdown("---")

    with map_col1:
        # ── Main scatter mapbox ───────────────────────────────
        plot_df = fdf.copy()
        # For schedule deviation, shift negative to 0 for size (negatives = ahead of schedule)
        if metric_col == 'Schedule_Deviation':
            plot_df['_size'] = plot_df[metric_col].clip(lower=0) + 1
        else:
            plot_df['_size'] = plot_df[metric_col].abs() + 0.01

        if map_color == "Risk Level":
            color_col = 'Risk_Level'
            color_map = RISK_COLORS
            color_seq = None
        else:
            color_col = 'Project_Type'
            color_map = None
            color_seq = px.colors.qualitative.Set2

        fig_map = px.scatter_mapbox(
            plot_df,
            lat='lat', lon='lon',
            color=color_col,
            size='_size',
            size_max=28,
            color_discrete_map=color_map,
            color_discrete_sequence=color_seq,
            hover_name='Project_ID',
            hover_data={
                'Project_Type': True,
                'Location': True,
                'Risk_Level': True,
                'Cost_Overrun': ':.0f',
                'Schedule_Deviation': ':.0f',
                'Safety_Risk_Score': ':.1f',
                'Completion_Percentage': ':.1f',
                'lat': False,
                'lon': False,
                '_size': False,
            },
            zoom=3.4,
            center={"lat": 38.5, "lon": -96.5},
            mapbox_style=map_style,
            opacity=0.80,
            title="Project Risk Map — USA",
        )

        # ── City label annotations ────────────────────────────
        if show_city_summary:
            for city_name, coords in CITY_COORDS.items():
                if city_name in fdf['Location'].values:
                    row = city_summary[city_summary['Location']==city_name]
                    if not row.empty:
                        r = row.iloc[0]
                        fig_map.add_trace(go.Scattermapbox(
                            lat=[coords['lat']],
                            lon=[coords['lon']],
                            mode='markers+text',
                            marker=dict(size=14, color='#2c3e50', symbol='circle'),
                            text=[city_name + "<br>" + str(int(r['Projects'])) + " projects"],
                            textposition='top right',
                            textfont=dict(size=11, color='#2c3e50'),
                            hoverinfo='skip',
                            showlegend=False,
                        ))

        fig_map.update_layout(
            height=560,
            margin=dict(t=40, b=5, l=0, r=0),
            legend=dict(orientation='h', y=-0.04, x=0, font_size=11),
        )
        st.plotly_chart(fig_map, use_container_width=True)

        # ── City-level choropleth (heatmap by metric) ─────────
        st.markdown("##### City-level averages")
        city_avg = fdf.groupby('Location').agg(
            Avg_Overrun_Pct=('Overrun_Pct','mean'),
            Avg_Delay=('Schedule_Deviation','mean'),
            High_Risk_Count=('Risk_Num', lambda x: (fdf.loc[x.index,'Risk_Level']=='High').sum()),
            Avg_Safety=('Safety_Risk_Score','mean'),
            Total_Accidents=('Accident_Count','sum'),
            Avg_Completion=('Completion_Percentage','mean'),
        ).reset_index().round(2)

        city_avg['lat'] = city_avg['Location'].map(lambda c: CITY_COORDS[c]['lat'])
        city_avg['lon'] = city_avg['Location'].map(lambda c: CITY_COORDS[c]['lon'])

        bubble_metric2 = st.selectbox("City bubble metric", [
            "Avg_Overrun_Pct","Avg_Delay","High_Risk_Count","Avg_Safety","Total_Accidents","Avg_Completion"
        ], format_func=lambda x: x.replace('_',' '))

        fig_city = px.scatter_mapbox(
            city_avg,
            lat='lat', lon='lon',
            size=bubble_metric2,
            size_max=55,
            color=bubble_metric2,
            color_continuous_scale='RdYlGn_r',
            hover_name='Location',
            hover_data={k: True for k in city_avg.columns if k not in ['lat','lon','Location']},
            text='Location',
            zoom=3.2,
            center={"lat": 38.5, "lon": -96.5},
            mapbox_style=map_style,
            title="City-Level Summary — " + bubble_metric2.replace('_',' '),
        )
        fig_city.update_traces(textposition='top center', textfont=dict(size=13, color='black'))
        fig_city.update_layout(
            height=420,
            margin=dict(t=40,b=5,l=0,r=0),
            coloraxis_colorbar=dict(title=bubble_metric2.replace('_',' '), thickness=14),
        )
        st.plotly_chart(fig_city, use_container_width=True)

# ══════════════════════════════════════════════════════════════
# TAB 3 — PROJECT SIMULATOR
# ══════════════════════════════════════════════════════════════
with tab3:
    st.markdown("### Project Risk Simulator")
    st.info("Adjust sliders to simulate a project. Get a risk prediction and AI decision plan from Ollama.")

    ca, cb, cc = st.columns(3)
    with ca:
        st.markdown("**Cost & Budget**")
        cost_overrun   = st.slider("Cost Overrun ($K)", -2000, 8000, int(df['Cost_Overrun'].mean()/1000), 100) * 1000
        overrun_pct    = st.slider("Overrun %", -20.0, 80.0, float(round(df['Overrun_Pct'].mean(),1)), 0.5)
    with cb:
        st.markdown("**Schedule**")
        sched_dev      = st.slider("Schedule Deviation (days)", -60, 400, int(df['Schedule_Deviation'].mean()), 5)
        sched_ratio    = st.slider("Schedule Ratio", 0.5, 2.5, float(round(df['Schedule_Ratio'].mean(),2)), 0.05)
        completion_pct = st.slider("Completion %", 0, 100, int(df['Completion_Percentage'].mean()), 1)
    with cc:
        st.markdown("**Safety & Operations**")
        safety_risk    = st.slider("Safety Risk Score", 0.0, 10.0, float(round(df['Safety_Risk_Score'].mean(),1)), 0.1)
        accidents      = st.slider("Accident Count", 0, 20, int(df['Accident_Count'].mean()), 1)
        vibration      = st.slider("Vibration Level", 0.0, 3.0, float(round(df['Vibration_Level'].mean(),2)), 0.05)
        crack_width    = st.slider("Crack Width (mm)", 0.0, 8.0, float(round(df['Crack_Width'].mean(),1)), 0.1)
        equip_util     = st.slider("Equipment Utilization %", 30.0, 100.0, float(round(df['Equipment_Utilization'].mean(),1)), 1.0)
        labor_hrs      = st.slider("Labor Hours", 2000, 15000, int(df['Labor_Hours'].mean()), 100)

    X_sim = pd.DataFrame([[cost_overrun, sched_dev, vibration, crack_width,
                           accidents, safety_risk, completion_pct,
                           equip_util, labor_hrs, overrun_pct, sched_ratio]], columns=FEATS)
    proba = model.predict_proba(X_sim)[0]
    label_map = {0:'High',1:'Low',2:'Medium'}
    pred_label = label_map[int(np.argmax(proba))]
    prob_dict = {label_map[i]: float(p) for i,p in enumerate(proba)}

    st.markdown("---")
    st.markdown("### Prediction")
    risk_icon = {'High':'🔴','Medium':'🟡','Low':'🟢'}
    m1,m2,m3,m4 = st.columns(4)
    m1.metric("Predicted Risk", risk_icon[pred_label] + " " + pred_label)
    m2.metric("High Risk Prob.", str(round(prob_dict.get('High',0)*100,1))+"%")
    m3.metric("Medium Risk Prob.", str(round(prob_dict.get('Medium',0)*100,1))+"%")
    m4.metric("Low Risk Prob.", str(round(prob_dict.get('Low',0)*100,1))+"%")

    risk_score = prob_dict.get('High',0)*100 + prob_dict.get('Medium',0)*50 + prob_dict.get('Low',0)*10
    gauge = go.Figure(go.Indicator(
        mode='gauge+number', value=round(risk_score,1),
        title={'text':'Composite Risk Score  (0 = safe, 100 = critical)'},
        gauge={'axis':{'range':[0,100]},'bar':{'color':'#3498db'},
               'steps':[{'range':[0,33],'color':'#27ae60'},
                        {'range':[33,66],'color':'#f39c12'},
                        {'range':[66,100],'color':'#e74c3c'}]}
    ))
    gauge.update_layout(height=260, margin=dict(t=50,b=5))
    st.plotly_chart(gauge, use_container_width=True)

    st.markdown("---")
    if st.button("Generate AI Decision Plan from Ollama", type='primary'):
        pct_str = str(round(max(proba)*100,1))
        prompt = (
            "You are a senior civil engineering project manager.\n\n"
            "Project metrics:\n"
            "- Predicted Risk: " + pred_label + " (confidence " + pct_str + "%)\n"
            "- Cost Overrun: $" + str(round(cost_overrun/1e6,2)) + "M (" + str(round(overrun_pct,1)) + "% over budget)\n"
            "- Schedule: " + str(sched_dev) + " days behind, ratio " + str(round(sched_ratio,2)) + "x, completion " + str(completion_pct) + "%\n"
            "- Safety Risk: " + str(round(safety_risk,1)) + "/10, Accidents: " + str(accidents) + "\n"
            "- Vibration: " + str(round(vibration,2)) + ", Crack Width: " + str(round(crack_width,1)) + "mm\n"
            "- Equipment Utilization: " + str(round(equip_util,0)) + "%, Labor Hours: " + str(labor_hrs) + "\n\n"
            "Give a decision plan with EXACTLY these sections:\n\n"
            "SITUATION:\n(2 sentences on what these numbers mean)\n\n"
            "OPTION 1 - COMPLETE EARLY:\n(3 concrete actions to accelerate delivery)\n\n"
            "OPTION 2 - REDUCE SCOPE:\n(3 specific things to cut or defer)\n\n"
            "RECOMMENDED DECISION:\n(Your recommended option and first 2 steps)\n\n"
            "RED FLAG:\n(The single most dangerous metric right now)\n\n"
            "Be direct and actionable. 1-2 sentences per point."
        )
        with st.spinner("Asking Ollama..."):
            resp = ask_ollama(prompt, model=ollama_model)
        st.markdown("#### AI Decision Plan")
        st.markdown(resp)
        st.markdown("---")
        st.markdown("#### Changes needed to reach Low Risk")
        actions = []
        if sched_dev > 30: actions.append("Cut schedule delay from **" + str(sched_dev) + " days** to under 30")
        if overrun_pct > 15: actions.append("Reduce cost overrun from **" + str(round(overrun_pct,1)) + "%** to under 15%")
        if accidents > 3: actions.append("Reduce accidents from **" + str(accidents) + "** to 3 or fewer")
        if safety_risk > 5: actions.append("Lower safety risk score from **" + str(round(safety_risk,1)) + "** to below 5")
        if completion_pct < 50: actions.append("Accelerate completion (currently only " + str(completion_pct) + "%)")
        if not actions:
            st.success("Already near Low Risk — maintain current controls.")
        else:
            for a in actions: st.markdown("- " + a)

# ══════════════════════════════════════════════════════════════
# TAB 4 — AI ADVISOR
# ══════════════════════════════════════════════════════════════
with tab4:
    st.markdown("### Portfolio-Level AI Advisor")
    st.info("Ask Ollama for portfolio-wide strategy based on your filtered projects.")
    high_df = fdf[fdf['Risk_Level']=='High']
    worst_city = fdf.groupby('Location')['Overrun_Pct'].mean().idxmax() if not fdf.empty else 'N/A'
    riskiest_type = high_df['Project_Type'].value_counts().idxmax() if not high_df.empty else 'N/A'
    s1,s2,s3,s4 = st.columns(4)
    s1.metric("High Risk Projects", str(len(high_df)))
    s2.metric("Avg Overrun", str(round(fdf['Overrun_Pct'].mean(),1))+"%")
    s3.metric("Worst City", worst_city)
    s4.metric("Riskiest Type", riskiest_type)
    question = st.text_area("Ask a specific question (optional)",
                            placeholder="e.g. Which 3 projects should I pause to save budget?", height=80)
    if st.button("Ask Ollama for Portfolio Strategy", type='primary'):
        n = len(fdf)
        nh = len(high_df)
        base = (
            "You are a data-driven civil engineering portfolio manager.\n\n"
            "Portfolio (" + str(n) + " projects):\n"
            "- High-risk: " + str(nh) + " (" + str(round(nh/max(n,1)*100,1)) + "%)\n"
            "- Avg cost overrun: " + str(round(fdf['Overrun_Pct'].mean(),1)) + "%\n"
            "- Avg schedule delay: " + str(round(fdf['Schedule_Deviation'].mean(),0)) + " days\n"
            "- Total accidents: " + str(int(fdf['Accident_Count'].sum())) + "\n"
            "- Avg completion: " + str(round(fdf['Completion_Percentage'].mean(),1)) + "%\n"
            "- Riskiest type: " + riskiest_type + "\n"
            "- Worst city for overruns: " + worst_city + "\n\n"
            + (("Question: " + question) if question else
               "Give 5 specific portfolio decisions to reduce risk and improve delivery speed. Number them. Be data-driven and concise.")
        )
        with st.spinner("Asking Ollama..."):
            resp = ask_ollama(base, model=ollama_model)
        st.markdown("#### Portfolio Strategy")
        st.markdown(resp)

st.markdown("---")
st.caption("BIM-AI Decision System | Streamlit + Plotly + Ollama | EM 622-A Stevens Institute")


## Launch Streamlit Dashboard

In [ ]:
import subprocess

proc = subprocess.Popen(
    ["streamlit", "run", "dashboard.py", "--server.port", "8501"],
)
print("✅ Dashboard running at: http://localhost:8501")
print("Open that URL in your browser.")
print("Stop with: proc.terminate()")